# Training the visual search model, start to finish

This notebook walks through the final model from this repository —
the one that predicts **where the eyes go next** during visual search
— and trains it from scratch **on the actual data**: 217,595 eye
movements from 333 people across 11 published experiments.

**Input and output.** For each eye-movement decision the model
receives: a picture of the display (pixels), the goal (the color and
shape to look for), where the eyes currently are, and the history of
previous trials. It returns **a probability for every item on the
screen**. One predicted saccade is one random draw.

**The model in one sentence.** Everything — what the display shows,
what you want, what you remember — is written into a single
**priority map**, read through an **attention window** centered on
your eyes, and turned into probabilities by a softmax.

**Before running:** build the dataset once (needs the public data
from https://osf.io/q27ph/):

    python pool_data.py --data_dir ".../Data Files" --out_dir dataset
    python build_contexts.py
    python build_contexts_v21.py

Needs `numpy`, `pandas`, `matplotlib`, `torch`. Training takes about
ten minutes on a laptop.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from front_end import render, shape_for, IMG
from build_contexts import item_positions
from build_contexts_v21 import (opponency_contrast, template_axis,
                                wedge_profiles, NBINS, MAXR)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

radii_np = np.linspace(0.09, MAXR, NBINS)   # distance bins along each ray
pos, _ = item_positions(6)                  # the six item slots (a ring)

## 1. The model, with a diagram

The equation, in words. For each item *i*:

    priority(i) = window(i) x [ goal-weighted evidence(i)
                                + target memory(i) - distractor memory(i)
                                - already-visited penalty(i) ]

    P(saccade -> i) = softmax over the items

Three sources flow into one map; the window (centered on the current
fixation) gates all of them; softmax reads the map out.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
boxes = [(0.03, "display\n(pixels)"),
         (0.19, "contrast\nmaps"),
         (0.35, "goal-weighted\nevidence"),
         (0.54, "+ memory\n(2 traces, IoR)"),
         (0.73, "x attention\nwindow"),
         (0.885, "softmax")]
for x, label in boxes:
    ax.add_patch(plt.Rectangle((x, 0.35), 0.13, 0.32, fc="#eef2fa",
                               ec="#334488", lw=1.5))
    ax.text(x + 0.065, 0.51, label, ha="center", va="center", fontsize=10)
for x, _ in boxes[:-1]:
    ax.annotate("", xy=(x + 0.17, 0.51), xytext=(x + 0.13, 0.51),
                arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("the goal: 'find the\ngreen diamond'", xy=(0.41, 0.67),
            xytext=(0.35, 0.92), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#118844"))
ax.annotate("previous trials", xy=(0.60, 0.67), xytext=(0.62, 0.92),
            ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#aa2222"))
ax.annotate("where the eyes are now", xy=(0.795, 0.35),
            xytext=(0.73, 0.10), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#884411"))
ax.set_xlim(0, 1.08); ax.set_ylim(0, 1); ax.axis("off")
plt.title("one priority map; the window gates everything; softmax reads it out")
plt.show()

## 2. Building it piece by piece

### 2a. From pixels to goal-directed evidence

We make one example display like the real experiments used: six items
of DIFFERENT shapes (feature search - the target is defined by its
shape but never "pops out" as the odd one), all green except one red
distractor; the target is the green diamond. We push it through the
perception stage: contrast maps, then the goal rotates the color axis so that
green scores positive and red negative. There is no "suppress red"
rule anywhere — red simply lands on the wrong side of the attend-green
axis.

In [ ]:
TARG, SING = 1, 4
items = [dict(x=pos[j][0], y=pos[j][1],
              color="red" if j == SING else "green",
              shape=shape_for(j + 1, TARG + 1)) for j in range(6)]
img = render(items)
maps = opponency_contrast(img)
u = template_axis("green")           # the goal, as a direction in color space

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(img, origin="lower")
axes[0].plot(IMG/2, IMG/2, "k+", ms=12)
axes[0].set_title("input: pixels (+ = fixation)")
v = np.abs(maps["RG"]).max()
axes[1].imshow(maps["RG"], origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axes[1].set_title("color contrast map")
D_T = u[0] * maps["RG"] + u[1] * maps["BY"]
v = np.abs(D_T).max()
axes[2].imshow(D_T, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axes[2].set_title("goal-rotated: green +, red -")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### A gallery: one example display per study

The dataset pools 11 experiments, and each used its own colors (and
one used four items instead of six). Below, one reconstructed display
per study, built exactly the way the training pipeline builds them —
target diamond among heterogeneous shapes, singleton in the study's
opposite color. (Needs `dataset/saccades_ctx.csv` from the build
steps; Hamblin-Frohman ships no color labels, so it falls back to
green/red.)

In [ ]:
gal = pd.read_csv("dataset/saccades_ctx.csv", low_memory=False,
                  usecols=["study", "setsize", "targCol", "singCol"])
gal = gal[gal.singCol != "none"]

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, (study, sub) in zip(axes.flat, gal.groupby("study")):
    setsize = int(sub.setsize.mode()[0])
    tc, sc = sub[["targCol", "singCol"]].mode().iloc[0]
    ipos, _ = item_positions(setsize)
    tg, sg = 2, (2 + setsize // 2) % setsize + 1   # target & singleton slots
    items = [dict(x=ipos[j][0], y=ipos[j][1],
                  color=sc if (j + 1) == sg else tc,
                  shape=shape_for(j + 1, tg)) for j in range(setsize)]
    ax.imshow(render(items), origin="lower")
    ax.set_title(f"{study}
{tc} target, {sc} singleton, "
                 f"set size {setsize}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[gal.study.nunique():]:
    ax.axis("off")
plt.suptitle("reconstructed example displays, one per study", y=0.995)
plt.tight_layout()
plt.show()

### 2b. Reading the maps from the eyes, through the window

The maps are sampled along rays fanning out from the current fixation
(like a radar sweep), giving each item a radial evidence profile. An
**attention window** — a sigmoid of distance whose steepness and
reach we will *learn* — weights near evidence more than far evidence.
Negative evidence is cut at zero: the red item is simply *relegated*,
not chased away.

For training, this perception stage is precomputed once for every
display in the dataset (`build_contexts_v21.py`); the model learns
only how to weight it.

In [ ]:
prof, _ = wedge_profiles(maps, u, (0.0, 0.0), 6)
prof = prof / prof.std(axis=(0, 1), keepdims=True)
window_demo = sigmoid(3.4 * (0.55 - radii_np))     # a window, pre-training

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for j, lab, c in [(TARG, "target", "#118844"),
                  (SING, "red distractor", "#aa2222"),
                  (0, "plain item", "#888888")]:
    axes[0].plot(radii_np, prof[j, :, 0], color=c, label=lab)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_xlabel("distance from fixation")
axes[0].set_title("goal-color evidence along each item's ray")
axes[0].legend()
axes[1].plot(radii_np, window_demo)
axes[1].set_xlabel("distance from fixation")
axes[1].set_title("an attention window (shape to be learned)")
plt.show()

### 2c. Memory of past trials

Two memories, one line each, updated after every trial: where targets
have been (pulls the eyes, fades fast) and where distractors have
been (pushes the eyes away, fades slowly). Their speeds (`eta`) and
strengths (`beta`) are learned. Within a trial, a third memory marks
items already inspected, so the eyes don't bounce back ("inhibition
of return", weight `g_I`).

    h = (1 - eta) * h          # everything fades a little
    h[location] += eta         # today's location gets a boost

## 3. The whole model in five lines

`P` holds each item's radial evidence profiles from the perception
stage (`D_T` = goal color, `D_O` = the sideways color axis, `D_P` =
brightness), `FORM` the shape evidence, `dist` each item's distance
from the current fixation, and `hT`, `hD`, `visited` the memories.
Eleven numbers to learn.

In [ ]:
def field(P, FORM, dist, visited, hT, hD, w, radii):
    win_ray = torch.sigmoid(w["k"] * (w["r0"] - radii))            # the window...
    mix = w["g_T"]*P[..., 0] + w["g_O"]*P[..., 1] + w["w_p"]*P[..., 2]
    stim = (torch.relu(mix) * win_ray).sum(-1) + w["g_form"]*FORM  # ...gates evidence
    win_item = torch.sigmoid(w["k"] * (w["r0"] - dist))            # ...and memory
    return stim + win_item * (w["beta_T"]*hT + w["beta_D"]*hD
                              + w["g_I"]*visited)                  # one priority map

print("that's the model. softmax(field) is the output.")

## 4. Train on the actual data

The dataset holds every scoreable saccade (indices 1-5) from 333
people: which item they looked at, the display they saw, where they
were fixating, and the full trial order (needed to rebuild each
person's memories).

We hold out 20% of the *people* — the model never sees them during
training — and fit by maximum likelihood: replay everyone's trials,
score the probability the model gives to each real eye movement, and
nudge all eleven numbers by gradient descent. The memory speeds are
learned too, so the memories are rebuilt inside the training loop.

In [ ]:
import fit_pooled as fp

sacc = pd.read_csv("dataset/saccades_ctx.csv", low_memory=False)
ev = pd.read_csv("dataset/events.csv", low_memory=False)
ctx = np.load("dataset/contexts_v21.npz")
sacc, tt = fp.build_tensors(sacc, ev)          # a few minutes: builds tensors

cid = torch.tensor(sacc.ctx.values.astype(int))
P = torch.tensor(ctx["P"])[cid]
FORM = torch.tensor(ctx["FORM"])[cid]
radii = torch.linspace(0.09, 1.1, P.shape[2])
N = len(sacc)
dist = torch.ones(N, 6)
for j in range(1, 7):
    ok = sacc[f"d{j}"].notna().values
    dist[ok, j-1] = torch.tensor(sacc.loc[ok, f"d{j}"].values,
                                 dtype=torch.float32)

n_subj = tt["eT"].shape[0]
rng = np.random.default_rng(0)
test_subj = torch.zeros(n_subj, dtype=torch.bool)
test_subj[rng.choice(n_subj, n_subj // 5, replace=False)] = True
test = test_subj[tt["si"]]
train = ~test
print(f"{N} saccades; {int((~test_subj).sum())} people to train on, "
      f"{int(test_subj.sum())} held out")

In [ ]:
def build_memories(eta_T, eta_D):
    # replay every subject's trials in order; return memory AS OF each trial
    S_, T_, _ = tt["eT"].shape
    hT = torch.zeros(S_, 6); hD = torch.zeros(S_, 6)
    outT, outD = [], []
    for t in range(T_):
        outT.append(hT); outD.append(hD)
        hT = (1 - eta_T) * hT + eta_T * tt["eT"][:, t]
        hD = (1 - eta_D) * hD + eta_D * tt["eD"][:, t]
    return torch.stack(outT, 1), torch.stack(outD, 1)

w = {name: torch.tensor(v, requires_grad=True) for name, v in
     [("g_T", 0.5), ("g_O", 0.0), ("w_p", 0.3), ("g_form", 1.0),
      ("k", 2.0), ("r0", 0.5), ("beta_T", 0.5), ("beta_D", -0.1),
      ("g_I", -0.5)]}
raw_eta = {n: torch.tensor(0.0, requires_grad=True) for n in ["T", "D"]}
optimizer = torch.optim.Adam(list(w.values()) + list(raw_eta.values()),
                             lr=0.05)

losses = []
for epoch in range(200):                       # ~10 minutes
    eta_T = torch.sigmoid(raw_eta["T"]); eta_D = torch.sigmoid(raw_eta["D"])
    memT, memD = build_memories(eta_T, eta_D)
    hT = memT[tt["si"], tt["ti"]]; hD = memD[tt["si"], tt["ti"]]
    F = field(P, FORM, dist, tt["visited"].float(), hT, hD, w, radii)
    F = F.masked_fill(~tt["valid"], -1e9)
    logp = torch.log_softmax(F, 1).gather(1, tt["choice"][:, None]).squeeze(1)
    loss = -logp[train].mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if epoch % 25 == 0:
        print(f"epoch {epoch}: loss {loss.item():.4f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("training step"); plt.ylabel("loss (per saccade)")
plt.title("training on the real eye movements")
plt.show()

In [ ]:
learned = {k: v.item() for k, v in w.items()}
learned["eta_T"] = torch.sigmoid(raw_eta["T"]).item()
learned["eta_D"] = torch.sigmoid(raw_eta["D"]).item()
for name, meaning in [
        ("g_T", "goal color attracts"),
        ("g_form", "goal shape attracts"),
        ("w_p", "any object attracts a little"),
        ("g_O", "sideways color axis (should be ~0)"),
        ("k", "window steepness"), ("r0", "window reach"),
        ("beta_T", "pull toward past target locations"),
        ("beta_D", "push from past distractor locations"),
        ("eta_T", "target memory speed (fast)"),
        ("eta_D", "distractor memory speed (slow)"),
        ("g_I", "penalty on already-visited items")]:
    print(f"{name:8} = {learned[name]:+7.3f}   {meaning}")

## 5. How good is it? (held-out people only)

We score the model ONLY on the people it never saw. Three
plain-language measures:

- **Probability on the true choice**: how much probability, on
  average, did the model put on the item the person actually looked
  at? (Chance: about 18%.)
- **Top-1 accuracy**: how often was the person's actual choice the
  model's single best guess?
- **Pseudo-R-squared**: 0 means no better than chance, 1 means
  perfect. For models of choices, 0.2-0.4 counts as excellent.

In [ ]:
with torch.no_grad():
    eta_T = torch.sigmoid(raw_eta["T"]); eta_D = torch.sigmoid(raw_eta["D"])
    memT, memD = build_memories(eta_T, eta_D)
    hT = memT[tt["si"], tt["ti"]]; hD = memD[tt["si"], tt["ti"]]
    F = field(P, FORM, dist, tt["visited"].float(), hT, hD, w, radii)
    F = F.masked_fill(~tt["valid"], -1e9)
    prob = torch.softmax(F, 1)
    p_true = prob.gather(1, tt["choice"][:, None]).squeeze(1)

nll = -p_true[test].log().mean()
chance = torch.log(tt["valid"].sum(1).float())[test].mean()
print(f"held-out saccades: {int(test.sum())}")
print(f"mean probability on the true choice: "
      f"{p_true[test].log().mean().exp()*100:.1f}%  "
      f"(chance {torch.exp(-chance)*100:.1f}%)")
print(f"top-1 accuracy: "
      f"{(prob.argmax(1) == tt['choice'])[test].float().mean()*100:.1f}%")
print(f"pseudo-R-squared vs chance: {1 - nll/chance:.3f}")

## 6. Does it behave like people? (held-out people only)

Numbers are one thing; the literature's signature *patterns* are the
real test. Using only the held-out people, we compare observed and
model rates for:

1. **Distractor suppression** — first saccades go to the red
   distractor *less often* than to an average plain item.
2. **Location priming** — saccades to the target roughly double when
   the target repeats its location; saccades to the distractor drop
   when the distractor repeats its location.

Nothing below was fitted to these specific patterns — they come out
of the one trained equation.

In [ ]:
first = torch.tensor((sacc.saccindex == 1).values)
sing_present = torch.tensor((sacc.singLoc > 0).values)
rows = torch.arange(N)
isT = torch.zeros(N, 6); isT[rows, torch.tensor(sacc.targLoc.values) - 1] = 1
isS = torch.zeros(N, 6)
sp = torch.tensor(sacc.singLoc.values)
isS[rows[sp > 0], sp[sp > 0] - 1] = 1
chose = torch.zeros(N, 6); chose[rows, tt["choice"]] = 1

m = test & first & sing_present
ot = (chose[m] * isT[m]).sum() / m.sum() * 100
os_ = (chose[m] * isS[m]).sum() / m.sum() * 100
mt = (prob[m] * isT[m]).sum() / m.sum() * 100
ms = (prob[m] * isS[m]).sum() / m.sum() * 100
n_plain = (tt["valid"][m].sum(1).float() - 2).clamp(min=1).mean()
ons = (100 - ot - os_) / n_plain
mns = (100 - mt - ms) / n_plain

x = np.arange(3); width = 0.36
plt.figure(figsize=(6.5, 3.5))
plt.bar(x - width/2, [ot, os_, ons], width, label="people", color="#444444")
plt.bar(x + width/2, [mt, ms, mns], width, label="model", color="#2233aa")
plt.xticks(x, ["target", "red\ndistractor", "plain item\n(average)"])
plt.ylabel("% of first saccades")
plt.title("suppression: the distractor is looked at LESS than a plain item")
plt.legend()
plt.show()

In [ ]:
ev2 = ev.copy()
ev2["block"] = pd.to_numeric(ev2["block"], errors="coerce").fillna(0.0)
ev2["trial"] = pd.to_numeric(ev2["trial"], errors="coerce")
ev2["subj"] = ev2["subj"].astype(str)
ev2 = ev2.sort_values(["study", "subj", "block", "trial"])
ev2["prevT"] = ev2.groupby(["study", "subj"]).targLoc.shift(1)
ev2["prevS"] = ev2.groupby(["study", "subj"]).singLoc.shift(1)
key = ["study", "subj", "block", "trial"]
sacc2 = sacc.merge(ev2[key + ["prevT", "prevS"]], on=key, how="left")

def bar_pair(masks, pick, labels, title, paper):
    obs = [((chose[m_] * pick[m_]).sum() / m_.sum() * 100).item()
           for m_ in masks]
    mod = [((prob[m_] * pick[m_]).sum() / m_.sum() * 100).item()
           for m_ in masks]
    x = np.arange(2)
    plt.bar(x - 0.18, obs, 0.36, label="people", color="#444444")
    plt.bar(x + 0.18, mod, 0.36, label="model", color="#2233aa")
    plt.xticks(x, labels); plt.legend()
    plt.title(f"{title}\n{paper}")

plt.figure(figsize=(11, 3.5))
plt.subplot(1, 2, 1)
tr = test & first & torch.tensor(sacc2.prevT.values == sacc.targLoc.values)
tc = test & first & torch.tensor((sacc2.prevT.values != sacc.targLoc.values)
                                 & ~np.isnan(sacc2.prevT.values))
bar_pair([tr, tc], isT,
         ["target location\nREPEATED", "target location\nchanged"],
         "% of first saccades to the TARGET", "(in the papers: 73 vs 37)")
plt.subplot(1, 2, 2)
sr = test & first & sing_present & torch.tensor(
    (sacc2.prevS.values == sacc.singLoc.values) & (sacc2.prevS.values > 0))
sc = test & first & sing_present & torch.tensor(
    sacc2.prevS.values != sacc.singLoc.values)
bar_pair([sr, sc], isS,
         ["distractor location\nREPEATED", "distractor location\nchanged"],
         "% of first saccades to the DISTRACTOR", "(in the papers: 5 vs 10)")
plt.tight_layout()
plt.show()

## Recap

| The model receives | It returns |
| --- | --- |
| a picture, a goal, the fixation, the trial history | a probability for every item |

We introduced the model and its diagram, built the pieces (perception
-> goal-weighted evidence -> window -> memory), wrote the whole model
in five lines, trained its eleven numbers on the real eye movements
by gradient descent, evaluated it on people it never saw (~29%
probability on the true choice against 18% chance; the right item is
its top guess about half the time), and watched it reproduce the
literature's signature patterns — distractor suppression and both
location-priming effects — from the one trained equation.

More: `RESULTS.md` (the full results record) and
`docs/priority_field_visual_search_model.md` (the theory).